# Forge Market EDA and Study Panel Selection

## 1. Purpose and scope

**Primary Analysis Question**: Can I detect market "anomalies" (whatever that might mean) in [EVE Online](https://www.eveonline.com/)'s trading marketplace?

I am selecting a fixed panel of item markets to study changes in price and trading activity in The Forge, the region containing EVE Online's most popular trading hub, Jita. Following the same markets over time should make it easier to study their behavior without constantly changing which items we're looking at.

I've decided that the *selection window* will be **June 1 through September 30, 2022**, inclusive, because this is a decently long period of time between any significant updates.

The proposed panel combines seven minerals with the top 50 manufactured Tech I products by market-history order count and the top 50 by estimated traded ISK. These items also have manufacturing relationships that we can use to investigate movements across related markets, rather than only looking at each item individually.

This notebook covers selecting these markets, checking their historical data availability, and getting a sense of their characteristics. We'll export the final panel for downstream analysis after documenting any exclusions.

I am also assuming that the latest retained reference data is not significantly different from the reference data throughout these few years apart from additions and published-status changes, since that would complicate the analysis significantly.

**Prerequisites for running this notebook**:
- Data ingested from Everef, covering at least the aforementioned selection window along with the reference data
- Running `mise run transform:build` to materialize the DuckDB database locally 

## 2. Data and reference preparation

### 2.1 Analytical setup

Market history and selection item references are accessed through the read-only DuckDB database.

The market history covers The Forge as a whole, not just transactions at Jita. It contains daily summaries of completed trading, rather than the standing buy and sell orders that we'd find in an order book.

In [1]:
import json
import math
import tarfile
from contextlib import contextmanager
from datetime import UTC, date, datetime

import ipywidgets as widgets
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import polars as pl
import pygwalker as pyg

# Use self-contained HTML output when JupyterLab does not render Plotly MIME output.
pio.renderers.default = "notebook"
pyg.GlobalVarManager.set_privacy("offline")
from IPython.display import HTML, clear_output, display

from transform.notebook_access import open_analysis_connection # access to the duckdb database

REGION_ID = 10000002
SELECTION_START = date(2022, 6, 1)
SELECTION_END = date(2022, 9, 30)
TOP_N = 50
SELECTION_DAYS = (SELECTION_END - SELECTION_START).days + 1

# Set SEED_OUTPUT_PATH before the export cell to redirect a validation run without changing the frozen seed.

# Shared category families: histograms use base hues; fine categories use related shades.
DISPLAY_GROUP_COLORS = {
    "Ships": "#2878B5",
    "Modules": "#E58326",
    "Ammunition and probes": "#38944A",
    "Deployables": "#C84343",
    "Drones": "#8555A6",
    "Minerals": "#555555",
    "Rigs": "#A88A26",
}
CATEGORY_COLORS = {
    "Battleships": "#205A8C",
    "Cruisers and battlecruisers": "#2878B5",
    "Freighters": "#589BC8",
    "Light ships": "#83B8DB",
    "Mining and industrial ships": "#396B87",
    "Defense and repair": "#B85E18",
    "Exploration and resource recovery": "#E58326",
    "Miscellaneous modules": "#EFA65B",
    "Mobility and concealment": "#C87943",
    "Ammunition and probes": "#38944A",
    "Deployables": "#C84343",
    "Drones": "#8555A6",
    "Minerals": "#555555",
    "Rigs": "#A88A26",
}
HISTOGRAM_CATEGORY_GROUPS = {
    "Battleships": "Ships",
    "Cruisers and battlecruisers": "Ships",
    "Freighters": "Ships",
    "Light ships": "Ships",
    "Mining and industrial ships": "Ships",
    "Defense and repair": "Modules",
    "Exploration and resource recovery": "Modules",
    "Miscellaneous modules": "Modules",
    "Mobility and concealment": "Modules",
    "Ammunition and probes": "Ammunition and probes",
    "Deployables": "Deployables",
    "Drones": "Drones",
    "Minerals": "Minerals",
    "Rigs": "Rigs",
}
HISTOGRAM_GROUP_ORDER = [
    "Ships", "Modules", "Ammunition and probes", "Deployables", "Drones", "Minerals", "Rigs",
]


### 2.2 Reference-data preparation

The market history tells us about prices and trading activity, but it doesn't tell us much about the items themselves or how they're manufactured.

For that, we're using [Everef's reference extract](https://docs.everef.net/datasets/reference-data), which was already obtained during the Airflow ingestion process. This gives us item names, Tech classifications, and manufacturing blueprints. We'll use these to identify our candidate products and, later on, connect them to their production inputs and outputs.

In [2]:
from pathlib import Path

# I'm not going to waste time editing the `ingestion` package just to get the blueprint.json data out
# Just gonna access the raw tarball real quick and just get it real quick
REFERENCE_ARCHIVE_DIRECTORY = Path(".local/data/raw/everef/reference-data/objects")
REFERENCE_ARCHIVE_SUFFIX = "_reference-data-latest.tar.xz"
REFERENCE_ARCHIVE_TIMESTAMP_FORMAT = "%Y%m%dT%H%M%SZ"


def find_repo_root(start_path=None):
    start = Path.cwd() if start_path is None else Path(start_path)
    for path in (start, *start.parents):
        if (path / "transformation" / "dbt_project.yml").is_file():
            return path
    raise FileNotFoundError(
        "Could not locate the repository root containing transformation/dbt_project.yml."
    )


def retained_reference_archive(repo_root=None):
    root = find_repo_root() if repo_root is None else Path(repo_root)
    archive_directory = root / REFERENCE_ARCHIVE_DIRECTORY
    candidates = []
    for archive_path in archive_directory.rglob(f"*{REFERENCE_ARCHIVE_SUFFIX}"):
        if not archive_path.is_file():
            continue
        timestamp_text, separator, _ = archive_path.name.partition("__")
        if separator != "__" or len(timestamp_text) != 16: # expected length of timestamp text
            continue
        try:
            parsed_timestamp = datetime.strptime(timestamp_text, REFERENCE_ARCHIVE_TIMESTAMP_FORMAT)
        except ValueError:
            continue
        if parsed_timestamp.strftime(REFERENCE_ARCHIVE_TIMESTAMP_FORMAT) != timestamp_text:
            continue
        candidates.append((timestamp_text, str(archive_path), archive_path))

    if not candidates:
        raise FileNotFoundError(
            f"No retained reference archive found under {archive_directory}. "
            "Run the reference ingestion workflow to retain a "
            "*_reference-data-latest.tar.xz archive before running this notebook."
        )

    _, _, archive_path = max(candidates)
    acquisition_timestamp = datetime.strptime(
        archive_path.name.partition("__")[0], REFERENCE_ARCHIVE_TIMESTAMP_FORMAT
    ).replace(tzinfo=UTC)
    return archive_path, acquisition_timestamp


def load_reference_records(archive_path):
    blueprints = None
    reference_types = None
    meta_groups = None
    with tarfile.open(archive_path, mode="r:xz") as archive:
        for member in archive:
            member_name = member.name.removeprefix("./")
            if member_name not in {"blueprints.json", "types.json", "meta_groups.json"} or not member.isfile():
                continue
            reference_file = archive.extractfile(member)
            if reference_file is None:
                raise ValueError(f"Reference extract could not read {member_name}.")
            with reference_file:
                reference_data = json.load(reference_file)
            if member_name == "blueprints.json":
                blueprints = reference_data
            elif member_name == "types.json":
                reference_types = reference_data
            else:
                meta_groups = reference_data

    if not isinstance(blueprints, dict) or not blueprints:
        raise ValueError("Reference extract must contain a nonempty blueprints.json object.")
    if not isinstance(reference_types, dict) or not reference_types:
        raise ValueError("Reference extract must contain a nonempty types.json object.")
    if not isinstance(meta_groups, dict) or not meta_groups:
        raise ValueError("Reference extract must contain a nonempty meta_groups.json object.")
    return blueprints, reference_types, meta_groups


reference_archive_path, reference_acquired_at = retained_reference_archive()
blueprints, reference_types, meta_groups = load_reference_records(reference_archive_path)

## 3. Candidate markets and selection-period activity

Before choosing the study panel, we'll need to identify the eligible manufactured products and compare their activity during the selection window. This should give us a sense of what we're selecting for and how.

### 3.1 Tech I candidates

We'll limit our activity analysis to items with an explicit [Tech I](https://wiki.eveuniversity.org/Tech_and_meta_levels) classification (`meta_group_id = 1` within the reference data) that also appear as manufacturing blueprint products. Tech I items are, broadly speaking, foundational goods in EVE Online:
- They offer basic, everyday ships, equipment, and consumables, used by both new and experienced players
- Some items are sold on the market by NPCs or offered as mission rewards, providing alternative supply sources alongside player manufacturing
- They have some of the simplest production chains in the game, mostly requiring the 7 primary minerals
- They require less [skill]() investment to use and produce than their Tech II counterparts
- Many of them are also used as production inputs for more advanced Tech II goods

These characteristics make Tech I products a very useful starting point for studying economic relationships in EVE Online's marketplace. Their relatively simple production chains also give us common inputs through which we can compare otherwise different item markets.

The rankings will only consider these manufactured products. We'll add the seven minerals separately later on so that their production inputs are represented as well.

Now, in order to make the analysis a bit more manageable, we're going to select the most active/economically interesting items and declare those as our **study panel**.

In [3]:
product_ids = set()
blueprints_without_products = []

for blueprint_id, blueprint in blueprints.items():
    if not isinstance(blueprint, dict) or not isinstance(blueprint.get("activities"), dict):
        raise ValueError(f"Invalid activities for blueprint {blueprint_id}.")
    activities = blueprint["activities"]

    # some listed blueprints are only used for *inventing* a new blueprint (i.e. Tech II blueprints), among other things
    if "manufacturing" not in activities:
        continue
    manufacturing = activities["manufacturing"]
    if not isinstance(manufacturing, dict):
        raise ValueError(f"Invalid manufacturing products for blueprint {blueprint_id}.")
    products = manufacturing.get("products")

    if products is None or products == {}:
        blueprints_without_products.append(int(blueprint_id))
        continue
    if not isinstance(products, dict):
        raise ValueError(f"Invalid manufacturing products for blueprint {blueprint_id}.")
    for product_key, product in products.items():
        type_id = product.get("type_id") if isinstance(product, dict) else None
        if type(type_id) is not int or type_id <= 0 or str(type_id) != product_key:
            raise ValueError(f"Invalid product {product_key} for blueprint {blueprint_id}.")
        product_ids.add(type_id)

if not product_ids:
    raise ValueError("No manufacturing products found in the reference extract.")

manufacturing_product_ids = pl.DataFrame(
    {"type_id": sorted(product_ids)}, schema={"type_id": pl.Int64}
)
manufacturing_reference_input = pl.DataFrame(
    {
        "Local archive path": [str(reference_archive_path)],
        "Acquired at (UTC)": [reference_acquired_at.isoformat()],
        "Blueprint records": [len(blueprints)],
        "Unique manufacturing products": [manufacturing_product_ids.height],
        "Blueprints without manufacturing products": [len(blueprints_without_products)],
    }
)
manufacturing_reference_input

In [4]:
@contextmanager
def open_study_connection():
    with open_analysis_connection() as con:
        con.register("manufacturing_product_ids", manufacturing_product_ids)
        yield con

### 3.2 Candidate activity relationships

In order to determine our most economically active/interesting items, we're going to aggregate the activity for each Tech I candidate during the aforementioned *selection window*, then select the most active ones. But before we do that, a quick overview of some of the base features in our data:

- Each row in the market history data represents a given market day, for a given *region*, and for a given *item* (AKA "type")
    - The grain is (`market_date`, `region_id`, `type_id`); note that we only care about The Forge region, already selected via the `dbt` pipeline
- Some of the notable features are:
    - `average`: the volume-weighted average price (VWAP) in ISK per unit
    - `highest`, `lowest`: the highest and lowest transaction prices in ISK per unit
    - `order_count`: the historical order count for that day, not the number of standing buy or sell orders
    - `volume`: the total number of item units traded
    - `estimated_traded_isk`: `average × volume`

Just to get a sense of what the aggregate candidate statistics will look like, we're calculating the following features:

- Average history orders per day: total history order count divided by all calendar days in the selection window
- Average units per order: total volume divided by total history order count
- Total estimated traded ISK: the sum of daily `average × volume`
- Average price over the selection period, or period VWAP: total estimated traded ISK divided by total volume

An item can trade frequently without being expensive, or trade relatively infrequently while still moving enormous amounts of ISK. These plots should help us see how those different measures relate before deciding how to rank the candidates.

In [5]:
activity_sql = """
with candidates as (
    select t.type_id, t.name_en as type_name
    from stg_everef_ref_types as t
    inner join manufacturing_product_ids as p on t.type_id = p.type_id
    where t.published = true and t.meta_group_id = 1
),

window_history as (
    select type_id, market_date, order_count, volume, average, estimated_traded_isk
    from int_forge_market_history
    where region_id = $region_id
        and market_date between $selection_start and $selection_end
),

activity_totals as (
    select
        c.type_id,
        c.type_name,
        count(h.market_date) as source_row_count,
        case when count(h.market_date) = 0 then 0
            else sum(h.order_count) end as total_order_count,
        case when count(h.market_date) = 0 then 0
            else sum(h.volume) end as total_volume,
        case when count(h.market_date) = 0 then 0
            else sum(h.estimated_traded_isk) end as total_estimated_traded_isk,
        case
            when sum(case when h.volume > 0 and h.average is null then 1 else 0 end) > 0 then null
            when sum(h.volume) = 0 then null
            else total_estimated_traded_isk / sum(h.volume)
        end as period_vwap
    from candidates as c
    left join window_history as h on c.type_id = h.type_id
    group by c.type_id, c.type_name
)
select *,
    total_volume / nullif(total_order_count, 0) as average_units_per_order
from activity_totals
"""

with open_study_connection() as con:
    activity = con.execute(
        activity_sql,
        {
            "region_id": REGION_ID,
            "selection_start": SELECTION_START,
            "selection_end": SELECTION_END,
        },
    ).pl()

if activity.is_empty() or activity.get_column("source_row_count").sum() == 0:
    raise ValueError("No candidate market history is available in the selection window.")

scatter_specs = [
    ("average_units_per_order", "Average units per order", ",.2f", "#3676a8"),
    ("period_vwap", "Period VWAP (ISK/unit)", ",.2f", "#c97c1d"),
    ("total_estimated_traded_isk", "Estimated traded value (ISK)", ",.0f", "#288779"),
]

relationship_activity = activity.with_columns(
    (pl.col("total_order_count").cast(pl.Float64) / SELECTION_DAYS).alias("average_history_orders_per_day")
)
relationship_series = [
    (
        metric, label, value_format, color,
        relationship_activity.filter(
            pl.col("average_history_orders_per_day").cast(pl.Float64).is_finite()
            & (pl.col("average_history_orders_per_day") > 0)
            & pl.col(metric).cast(pl.Float64).is_finite()
            & (pl.col(metric) > 0)
        ),
    )
    for metric, label, value_format, color in scatter_specs
]

In [6]:
def plot_candidate_relationships(
    series, *, title, x_metric, x_label, eligibility_minimum, eligibility_label,
):
    row_count = len(series)
    plot_height = 300 * row_count + 40 * (row_count - 1)
    x_values = [eligibility_minimum] + [
        value for *_, plotted in series for value in plotted.get_column(x_metric).to_list()
    ]
    log_min, log_max = math.log10(min(x_values)), math.log10(max(x_values))
    padding = max((log_max - log_min) * 0.05, 0.05)
    x_range = [log_min - padding, log_max + padding]
    fig = make_subplots(
        rows=row_count,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=40 / plot_height,
    )
    for row, (metric, label, value_format, color, plotted) in enumerate(series, start=1):
        fig.add_trace(
            go.Scattergl(
                x=plotted.get_column(x_metric).to_list(),
                y=plotted.get_column(metric).to_list(),
                mode="markers",
                marker={"color": color, "size": 7, "opacity": 0.65},
                customdata=plotted.select(
                    "type_name", "type_id", "total_volume", "total_order_count",
                ).rows(),
                hovertemplate=(
                    "<b>%{customdata[0]}</b> (type %{customdata[1]})<br>"
                    + f"{x_label}: %{{x:,.4g}}<br>"
                    + "Total history order count: %{customdata[3]:,.0f}<br>"
                    + ("Total volume: %{customdata[2]:,.0f}<br>" if row == 1 else "")
                    + f"{label}: %{{y:{value_format}}}<extra></extra>"
                ),
                showlegend=False,
            ),
            row=row,
            col=1,
        )
        # Anchor the real shared axis at the bottom for native hover placement.
        # Overlay axes supply only the upper rows' ticks, without empty hover panels.
        tick_row = row_count if row == 1 else row - 1
        tick_anchor = "y" if tick_row == 1 else f"y{tick_row}"
        fig.update_xaxes(
            type="log", range=x_range, showticklabels=True, tickmode="auto",
            anchor=tick_anchor,
            matches="x" if row > 1 else None,
            overlaying="x" if row > 1 else None,
            title=x_label if row == 1 else None,
            row=row, col=1,
        )
        fig.update_yaxes(type="log", title=label, anchor="x", row=row, col=1)
        fig.add_vline(x=eligibility_minimum, line_dash="dash", line_color="#777777", row=row, col=1)

    # Cross-panel hover needs the same trace axis, not only matched ranges.
    fig.update_traces(xaxis="x")
    fig.add_annotation(
        text=eligibility_label, x=0, y=1.025, xref="paper", yref="paper",
        xanchor="left", showarrow=False,
    )
    fig.update_layout(
        title=title,
        width=1300,
        height=plot_height + 155,
        hovermode="closest",
        hoverlabel={"showarrow": True},
        margin={"t": 75, "r": 20, "b": 80, "l": 75},
    )
    return fig

In [7]:
plot_candidate_relationships(
    relationship_series,
    title="Candidate activity relationships - aggregates over selection window",
    x_metric="average_history_orders_per_day",
    x_label="Average history orders per day",
    eligibility_minimum=1,
    eligibility_label="ISK-ranking eligibility minimum",
)

The three panels compare average units per order, period VWAP, and estimated traded value against average history orders per day. That daily average uses all 122 calendar days in the selection window, not just the days with recorded activity.

Both axes are logarithmic. Each panel includes candidates with positive, finite values for the measures being shown. The dashed line at 1 history order per day marks the minimum activity we'll require for the estimated-ISK ranking later on.

A few things stand out:

- **Average units per order:** Most candidates trade only a few units per order, even among the more frequently traded items. The upper band broadly consists of consumables like Antimatter Charge S and Scourge Light Missile, which are bought and sold in much larger quantities.
- **Period VWAP:** There is a negative relationship on the log scales. More frequently traded items tend to be cheaper, while some extremely expensive items trade very infrequently.
- **Estimated traded value:** There is a positive relationship on the log scales, although items with similar order frequencies can still differ by orders of magnitude in traded value. Some relatively infrequently traded items also move substantial amounts of ISK.

This gives us a reason to use both history order count and estimated traded ISK when choosing the panel, since they each capture different kinds of market activity.

**TODO:** Add fitted lines and Pearson correlations of the logged values to the period-VWAP and estimated-traded-value panels.

### 3.3 Selection rankings

We're going to rank the candidates separately by two measures over the selection window:

- **History order count:** the sum of daily market-history `order_count`
- **Estimated traded ISK:** the sum of daily VWAP multiplied by daily traded volume

The first should pick out frequently traded items, while the second should also capture expensive items that account for substantial amounts of traded ISK.

For the ISK ranking, I'll require at least one history order per day on average over the selection window. That's a minimum of 122 total history orders, rather than a requirement that the item trades every single day. This helps avoid selecting items that have very rare trades but also cost enormous amounts of ISK; removing the items with sparse activity will make the analysis less cumbersome.

After this ranking, we will simply select the top 50 from each set of candidates.

In [8]:
ranking_sql = """
with isk_ranks as (
    select type_id,
        rank() over (order by total_estimated_traded_isk desc nulls last) as estimated_traded_isk_rank
    from candidate_activity
    where total_order_count >= $selection_days
)
select
    a.*,
    rank() over (order by a.total_order_count desc nulls last) as order_count_rank,
    i.estimated_traded_isk_rank
from candidate_activity as a
left join isk_ranks as i on a.type_id = i.type_id
"""

with open_study_connection() as con:
    con.register("candidate_activity", activity)
    activity = con.execute(ranking_sql, {"selection_days": SELECTION_DAYS}).pl()

top_by_order_count = activity.filter(pl.col("order_count_rank") <= TOP_N).sort(
    "total_order_count", descending=True
)
top_by_estimated_isk = activity.filter(pl.col("estimated_traded_isk_rank") <= TOP_N).sort(
    "total_estimated_traded_isk", descending=True
)

### 3.4 Activity distributions and selection cutoffs

Before looking at the individual ranked items, let's see where the top-50 cutoffs fall within the candidate distributions.

Each cumulative curve shows the percentage of candidates with totals at or below a given value. The horizontal axis is logarithmic, and the dashed line marks the lowest selected total.

The history-order chart uses all candidates with positive, finite order totals. The estimated-ISK chart uses the eligible candidates that pass our minimum order-count requirement and have positive, finite estimated traded values.

In [9]:
order_distribution_positive = activity.with_columns(
    pl.col("total_order_count").cast(pl.Float64)
).filter(pl.col("total_order_count").is_finite() & (pl.col("total_order_count") > 0))
order_cutoff = top_by_order_count.get_column("total_order_count").min()

isk_distribution_eligible = activity.filter(pl.col("estimated_traded_isk_rank").is_not_null())
isk_distribution_positive = isk_distribution_eligible.with_columns(
    pl.col("total_estimated_traded_isk").cast(pl.Float64)
).filter(
    pl.col("total_estimated_traded_isk").is_finite() & (pl.col("total_estimated_traded_isk") > 0)
)
isk_cutoff = top_by_estimated_isk.get_column("total_estimated_traded_isk").min()

In [10]:
def plot_activity_distribution(
    positive, *, metric, cutoff, cutoff_text, top_n, title, x_label, y_label, unit, color,
):
    if positive.is_empty() or cutoff is None or not math.isfinite(cutoff) or cutoff <= 0:
        raise ValueError(f"No positive finite totals or selected cutoff for {title}.")
    fig = px.ecdf(
        positive, x=metric, ecdfnorm="percent", log_x=True,
        color_discrete_sequence=[color], hover_data=["type_id", "type_name"], title=title,
    )
    fig.update_traces(hovertemplate=(
        "<b>%{customdata[1]}</b> (type %{customdata[0]})<br>"
        + f"{x_label}: %{{x:,.0f}}<br>"
        + f"{y_label}: %{{y:.2f}}<extra></extra>"
    ))
    fig.add_shape(
        type="line", x0=cutoff, x1=cutoff, y0=0, y1=1, xref="x", yref="y domain",
        line={"dash": "dash", "color": "#4d4d4d", "width": 2},
    )
    fig.add_annotation(
        x=math.log10(cutoff), y=20, xref="x", yref="y",
        text=f"Top-{top_n} cutoff<br>{cutoff_text}", showarrow=False,
        hovertext=f"Top-{top_n} cutoff: {cutoff:,.0f} {unit}", captureevents=True,
        xanchor="right", yanchor="top", font={"color": "white"},
        bgcolor="#4d4d4d", bordercolor="white", borderwidth=1,
    )
    fig.update_xaxes(title=x_label, exponentformat="B")
    fig.update_yaxes(title=y_label, range=[0, 102])
    fig.update_layout(height=450, width=1200, showlegend=False)
    return fig

In [11]:
plot_activity_distribution(
    order_distribution_positive, metric="total_order_count", cutoff=order_cutoff,
    cutoff_text=f"{order_cutoff:,.0f} orders" if order_cutoff is not None else "", top_n=TOP_N,
    title="History order count cumulative distribution", x_label="History order count",
    y_label="Cumulative share of candidates (%)", unit="history orders", color="#3676a8",
)

The top-50 cutoff is **30,461 history orders** over the selection window, or roughly **250 per calendar day on average**. This puts our selection well into the upper tail of the candidate distribution at ~95.5% and above.

There isn't an obvious break at this point that separates active markets from everything else. The top 50 is a practical choice for keeping the study manageable, rather than a naturally distinct group of items.

In [12]:
plot_activity_distribution(
    isk_distribution_positive, metric="total_estimated_traded_isk", cutoff=isk_cutoff,
    cutoff_text=f"{isk_cutoff / 1e9:,.1f}B ISK" if isk_cutoff is not None else "", top_n=TOP_N,
    title="Estimated traded value cumulative distribution", x_label="Estimated traded value (ISK)",
    y_label="Cumulative share of eligible candidates (%)", unit="ISK", color="#288779",
)

The estimated-traded-value cutoff is approximately **120.2 billion ISK** over the selection window. We're again selecting from the upper tail at roughly 94.3%, although this curve only includes candidates that pass the minimum order-count requirement.

As with the history-order distribution, there isn't a sharp separation at the cutoff. Together, these selections give us a deliberately high-activity panel, rather than a representative sample of all Tech I markets.

### 3.5 Ranked candidates

Now let's look at which items actually made it into each ranking. The charts use separate scales, with type IDs included in the labels to distinguish items that share a name.

In [13]:
order_ranking_data = top_by_order_count.with_columns(
    pl.concat_str(
        [
            pl.col("type_name").fill_null("Name unavailable"),
            pl.lit(" ["),
            pl.col("type_id").cast(pl.String),
            pl.lit("]"),
        ]
    ).alias("label")
)

isk_ranking_data = top_by_estimated_isk.with_columns(
    pl.concat_str(
        [
            pl.col("type_name").fill_null("Name unavailable"),
            pl.lit(" ["),
            pl.col("type_id").cast(pl.String),
            pl.lit("]"),
        ]
    ).alias("label")
)

In [14]:
def plot_activity_ranking(chart_data, *, metric, title, x_label, color):
    fig = px.bar(
        chart_data, x=metric, y="label", orientation="h", color_discrete_sequence=[color],
        hover_data={"type_id": True, "type_name": True, "order_count_rank": True,
                    "estimated_traded_isk_rank": True, metric: ":,.0f"},
        title=title,
    )
    fig.update_yaxes(autorange="reversed", title=None)
    fig.update_xaxes(title=x_label, rangemode="tozero", exponentformat="B")
    fig.update_layout(height=max(500, chart_data.height * 27), width=1200, showlegend=False)
    return fig

In [15]:
plot_activity_ranking(
    order_ranking_data, metric="total_order_count", title=f"Top {TOP_N} by history order count",
    x_label="History order count", color="#3676a8",
)

In terms of the most ordered items, these primarily comprise fairly cheap equipment and consumable items, as one might expect. These include [ship modules](https://wiki.eveuniversity.org/Modules), [probes](https://wiki.eveuniversity.org/Probe_scanning), [drones](https://wiki.eveuniversity.org/Drones), and [missiles](https://wiki.eveuniversity.org/Missiles). Some of these also include relatively cheap beginner ships like the [Heron](https://wiki.eveuniversity.org/Heron) and [Venture](https://wiki.eveuniversity.org/Venture).

Exploration equipment is particularly prominent here. Core Probe Launcher I and Core Scanner Probe I lead the ranking, with analyzers, salvagers, and other exploration-related modules also appearing near the top.

In [16]:
plot_activity_ranking(
    isk_ranking_data, metric="total_estimated_traded_isk", title=f"Top {TOP_N} by estimated traded value",
    x_label="Estimated traded value (ISK)", color="#288779",
)

Most of the total traded ISK is attributed to sales of various ships, most notably [freighters](https://wiki.eveuniversity.org/Freighters) and [battlecruisers](https://wiki.eveuniversity.org/Battlecruisers)/[battleships](https://wiki.eveuniversity.org/Battleships). These are expensive ships that often go through significant churn.

Freighters are large ships used to freight enormous quantities of cargo. Although they are often only used in safe "high-sec" (high security) space, they are often targeted due to them carrying copious amounts of valuable loot. As such, the constant destruction and replacement of these valuable ships results in substantial amounts of traded ISK.

Battlecruisers/battleships are also expensive ships and, as the name implies, often face destruction in PvE and PvP scenarios. Replacing these ships similarly creates ongoing demand.

## 4. Proposed panel

I'm combining the two top-50 lists with the seven standard mineral inputs: Tritanium, Pyerite, Mexallon, Isogen, Nocxium, Zydrine, and Megacyte. After removing duplicate items, this gives us a proposed study panel that covers both frequently traded products and products that account for substantial traded ISK, along with their primary mineral inputs. 

I'll also assign categories manually to make the resulting panel easier to describe and compare.

In [17]:
MINERAL_IDS = [34, 35, 36, 37, 38, 39, 40]

membership = pl.concat(
    [
        pl.DataFrame({"type_id": MINERAL_IDS, "selection_reason": "curated mineral"}),
        top_by_order_count.select("type_id").with_columns(
            pl.lit("top 50 history order count").alias("selection_reason")
        ),
        top_by_estimated_isk.select("type_id").with_columns(
            pl.lit("top 50 estimated traded ISK").alias("selection_reason")
        ),
    ]
).cast({"type_id": pl.Int64, "selection_reason": pl.String})

panel_sql = """
with members as (
    select
        type_id,
        bool_or(selection_reason = 'curated mineral') as is_mineral,
        bool_or(selection_reason = 'top 50 history order count') as in_order_count_top_50,
        bool_or(selection_reason = 'top 50 estimated traded ISK') as in_estimated_isk_top_50,
        string_agg(selection_reason, '; ' order by selection_reason) as selection_reason
    from membership
    group by type_id
),
window_totals as (
    select
        m.type_id,
        case when count(h.market_date) = 0 then 0
            else sum(h.order_count) end as total_order_count,
        case when count(h.market_date) = 0 then 0
            else sum(h.estimated_traded_isk) end as total_estimated_traded_isk
    from members as m
    left join int_forge_market_history as h
        on m.type_id = h.type_id
        and h.region_id = $region_id
        and h.market_date between $selection_start and $selection_end
    group by m.type_id
)
select
    m.*,
    t.name_en as type_name,
    a.order_count_rank,
    a.estimated_traded_isk_rank,
    w.total_order_count,
    w.total_estimated_traded_isk
from members as m
left join stg_everef_ref_types as t on m.type_id = t.type_id
left join activity as a on m.type_id = a.type_id
left join window_totals as w on m.type_id = w.type_id
"""

with open_study_connection() as con:
    con.register("membership", membership)
    con.register("activity", activity)
    proposed_panel = con.execute(
        panel_sql,
        {"region_id": REGION_ID, "selection_start": SELECTION_START, "selection_end": SELECTION_END},
    ).pl()

if proposed_panel.get_column("type_id").n_unique() != proposed_panel.height:
    raise ValueError("The proposed panel contains duplicate type IDs.")
if not set(MINERAL_IDS).issubset(set(proposed_panel.get_column("type_id").to_list())):
    raise ValueError("The proposed panel is missing a curated mineral.")
if not set(top_by_order_count.get_column("type_id").to_list()).issubset(
    set(proposed_panel.filter(pl.col("in_order_count_top_50")).get_column("type_id").to_list())
):
    raise ValueError("The proposed panel is missing an order-count selection.")
if not set(top_by_estimated_isk.get_column("type_id").to_list()).issubset(
    set(proposed_panel.filter(pl.col("in_estimated_isk_top_50")).get_column("type_id").to_list())
):
    raise ValueError("The proposed panel is missing an estimated-ISK selection.")

proposed_panel = proposed_panel.with_columns(
    pl.lit(REGION_ID).alias("region_id"),
    pl.lit(SELECTION_START, dtype=pl.Date).alias("selection_start"),
    pl.lit(SELECTION_END, dtype=pl.Date).alias("selection_end"),
)

### 4.1 Membership categories and ranked-selection overlap

First, let's see how much the two rankings actually overlap. The bar below separates items selected only by history order count, only by estimated traded ISK, or by both rankings.

The percentages describe the unique union of the ranked lists. The manually added minerals aren't included in this comparison.

In [18]:
ranked_union = proposed_panel.filter(
    pl.col("in_order_count_top_50") | pl.col("in_estimated_isk_top_50")
)
ranked_union_count = ranked_union.height
if ranked_union_count == 0:
    raise ValueError("The ranked-selection union is empty.")

overlap_segments = [
    ("Order-only", pl.col("in_order_count_top_50") & ~pl.col("in_estimated_isk_top_50"), "#3676a8"),
    ("Both", pl.col("in_order_count_top_50") & pl.col("in_estimated_isk_top_50"), "#7a5195"),
    ("ISK-only", ~pl.col("in_order_count_top_50") & pl.col("in_estimated_isk_top_50"), "#288779"),
]
overlap_counts = [
    (label, ranked_union.filter(predicate).height, color)
    for label, predicate, color in overlap_segments
]
if sum(count for _, count, _ in overlap_counts) != ranked_union_count:
    raise ValueError("Ranked-selection overlap does not partition the unique union.")

curated_only_additions = proposed_panel.filter(
    pl.col("is_mineral")
    & ~pl.col("in_order_count_top_50")
    & ~pl.col("in_estimated_isk_top_50")
).height

overlap_shares = [
    (label, count, count / ranked_union_count, color)
    for label, count, color in overlap_counts
]

In [19]:
def plot_ranked_selection_overlap(segments, *, curated_only_count, title, x_label):
    fig = go.Figure()
    for label, count, share, color in segments:
        fig.add_trace(go.Bar(
            y=[""], x=[count], name=label, orientation="h", marker_color=color,
            text=[f"{count:,} ({share:.1%})"], textposition="inside", insidetextanchor="middle",
            hovertemplate=(
                f"{label}<br>{count:,} items<br>"
                f"{share:.1%} of unique ranked union<extra></extra>"
            ),
        ))
    fig.update_layout(
        title=f"{title}<br><sup>Curated-only additions: {curated_only_count:,}</sup>",
        barmode="stack", xaxis_title=x_label, yaxis_title=None,
        width=750, height=240, margin={"t": 65, "b": 75, "l": 20, "r": 20},
        legend={"title_text": "", "orientation": "h", "x": 0.5, "xanchor": "center",
                "y": -0.5, "yanchor": "top"},
    )
    fig.update_yaxes(showticklabels=False)
    return fig

In [20]:
plot_ranked_selection_overlap(
    overlap_shares, curated_only_count=curated_only_additions,
    title="Ranked-selection overlap", x_label="Number of items",
)

Only **5 items** appear in both rankings, with each ranking contributing **45 items** that don't appear in the other. That gives us **95 unique ranked items**, or **102 proposed items** after adding the seven minerals.

This shows a pretty substantial difference between the selections, and further validates that our selection method guarantees that we're analyzing a broader set of markets that we'd get from choosing either measure alone.

In [21]:
category_ids = {
    "Minerals": [34, 35, 36, 37, 38, 39, 40],
    "Ammunition and probes": [209, 210, 211, 212, 213, 215, 216, 217, 218, 219, 220, 221, 222, 230, 22778, 30013],
    "Drones": [2183, 2203, 2454, 2464, 2486, 15508],
    "Deployables": [33474, 33475, 33700, 33702],
    "Structures": [35833],
    "Rigs": [25894, 26929, 30993, 31213, 31358, 31360, 31370, 31716, 31718, 31788, 31790],
    "Light ships": [605, 672, 49710, 54731],
    "Cruisers and battlecruisers": [621, 622, 626, 627, 16227, 16229, 16231, 24696, 24698, 24700, 24702, 47270, 54732],
    "Battleships": [638, 639, 640, 641, 642, 643, 644, 645],
    "Mining and industrial ships": [2998, 17476, 17478, 17480, 32880],
    "Freighters": [20183, 20185, 20187, 20189],
    "Mobility and concealment": [434, 439, 21857, 33985, 11370, 34126],
    "Defense and repair": [578, 2046, 3829, 4403, 40349, 41498],
    "Exploration and resource recovery": [17938, 22175, 22177, 25861, 33180, 17482, 25266],
    "Miscellaneous modules": [23527, 24283, 11563, 41492],
}

assignments = pl.DataFrame(
    [(type_id, category) for category, ids in category_ids.items() for type_id in ids],
    schema=["type_id", "study_category"],
    orient="row",
).cast({"type_id": pl.Int64, "study_category": pl.String})

if assignments.get_column("type_id").n_unique() != assignments.height:
    raise ValueError("A type ID has more than one study category.")

study_panel = proposed_panel.select("type_id", "type_name").join(
    assignments, on="type_id", how="left", validate="1:1"
)

unassigned = study_panel.filter(
    pl.col("study_category").is_null()
).select("type_id", "type_name")

if not unassigned.is_empty():
    raise ValueError("Assign a study category to each proposed item before reviewing history.")

### 4.2 Suitability: historical source-row coverage

Now that we have a proposed panel, let's check how much historical data is available for these items beyond the selection window.

The heatmap shows the percentage of calendar days in each month with a recorded market-history row for each item. A recorded day with zero activity still counts as recorded; we're checking whether the row exists, rather than how much trading occurred.

The incomplete final month is omitted so that we're comparing complete months.

In [22]:
history_summary_sql = """
select
    p.type_id,
    p.type_name,
    min(h.market_date) as first_observed_date,
    max(h.market_date) as last_observed_date,
    count(h.market_date) as source_row_count,
    count(distinct h.market_date) as observed_day_count
from notebook_study_panel as p
left join int_forge_market_history as h
    on p.type_id = h.type_id
    and h.region_id = $region_id
    and h.market_date >= $history_start
group by p.type_id, p.type_name
"""

monthly_presence_sql = """
select
    h.type_id,
    cast(date_trunc('month', h.market_date) as date) as month,
    count(distinct h.market_date) as recorded_day_count
from int_forge_market_history as h
inner join notebook_study_panel as p on h.type_id = p.type_id
where h.region_id = $region_id and h.market_date >= $history_start
group by h.type_id, date_trunc('month', h.market_date)
"""

if study_panel.get_column("type_id").n_unique() != study_panel.height:
    raise ValueError("Study-panel membership must be unique before reviewing history.")

with open_analysis_connection() as con:
    con.register("notebook_study_panel", study_panel)
    parameters = {"region_id": REGION_ID, "history_start": SELECTION_START}
    history_summary = con.execute(history_summary_sql, parameters).pl()
    monthly_presence = con.execute(monthly_presence_sql, parameters).pl()

if history_summary.height != study_panel.height or history_summary.get_column("type_id").n_unique() != history_summary.height:
    raise ValueError("The history review must retain one row for each study-panel item.")
if history_summary.filter(pl.col("source_row_count") != pl.col("observed_day_count")).height:
    raise ValueError("Observed history contains duplicate item-date source rows.")
if monthly_presence.select(["type_id", "month"]).n_unique() != monthly_presence.height:
    raise ValueError("Monthly source-row presence must have one row per item-month.")

available_cutoff_value = history_summary.get_column("last_observed_date").drop_nulls().max()
if available_cutoff_value is None:
    raise ValueError("No observed history is available for the proposed panel.")
available_cutoff = date.fromisoformat(str(available_cutoff_value))
full_history_days = (available_cutoff - SELECTION_START).days + 1
if full_history_days <= 0:
    raise ValueError("The available history cutoff precedes the declared interval.")

if history_summary.filter(pl.col("observed_day_count") > full_history_days).height:
    raise ValueError("Observed-day coverage cannot exceed the declared interval.")

months = pl.date_range(SELECTION_START, available_cutoff, interval="1mo", eager=True)
# Include the final month only when its last calendar day has been observed.
months = months.filter(months.dt.month_end() <= available_cutoff)
panel_order = study_panel.select("type_id").with_row_index("panel_order")
presence_grid = (
    panel_order.join(pl.DataFrame({"month": months}), how="cross")
    .join(monthly_presence, on=["type_id", "month"], how="left", validate="1:1")
    .with_columns(
        pl.col("recorded_day_count").fill_null(0),
        pl.col("month").dt.month_end().dt.day().alias("expected_day_count"),
    )
    .with_columns(
        (pl.col("recorded_day_count") / pl.col("expected_day_count") * 100)
        .alias("recorded_day_pct")
    )
    .sort(["panel_order", "month"])
)
if presence_grid.filter(pl.col("recorded_day_count") > pl.col("expected_day_count")).height:
    raise ValueError("Monthly recorded days cannot exceed expected calendar days.")
presence_matrix = presence_grid.get_column("recorded_day_pct").to_numpy().reshape(
    study_panel.height, months.len()
)
presence_hover = presence_grid.select("recorded_day_count", "expected_day_count").to_numpy().reshape(
    study_panel.height, months.len(), 2
)
labels = study_panel.select(
    pl.concat_str(
        [
            pl.col("type_name").fill_null("Name unavailable"),
            pl.lit(" ["),
            pl.col("type_id").cast(pl.String),
            pl.lit("]"),
        ]
    ).alias("label")
).get_column("label").to_list()

In [23]:
def plot_monthly_history_availability(
    percentages, *, months, labels, recorded_expected_days, colorscale,
    title, x_label, colorbar_label,
):
    fig = go.Figure(go.Heatmap(
        z=percentages,
        x=months,
        y=labels,
        customdata=recorded_expected_days,
        colorscale=colorscale,
        zmin=0,
        zmax=100,
        colorbar={"title": colorbar_label},
        hovertemplate=(
            "<b>%{y}</b><br>%{x|%Y-%m}<br>"
            "%{customdata[0]:.0f} of %{customdata[1]:.0f} days recorded — %{z:.1f}%"
            "<extra></extra>"
        ),
    ))
    fig.update_yaxes(autorange="reversed", title=None)
    fig.update_xaxes(dtick="M3", tickformat="%Y-%m", tickangle=-45, title=x_label)
    fig.update_layout(height=max(600, len(labels) * 22), width=1500, title=title)
    return fig

In [24]:
plot_monthly_history_availability(
    presence_matrix,
    months=months.to_list(),
    labels=labels,
    recorded_expected_days=presence_hover,
    colorscale="Blues",
    title="Monthly market-history availability",
    x_label="Month",
    colorbar_label="Recorded days (%)",
)

Most of the proposed items have recorded history on nearly every day throughout most months. There are some isolated gaps, along with a shared reduction in September 2023 that coincides with EVE Online's [unplanned outage](https://forums.eveonline.com/t/resolved-unexpected-downtime-1st-september-2023/418301).

Two items stand out for having consistently fewer recorded days: the **Prototype Jump Drive Economizer** and the **Fortizar**.

### 4.3 Exclusions

The [Prototype Jump Drive Economizer](https://wiki.eveuniversity.org/Capital_Ship_Modules) is a Tech I module primarily used for Tech II [Jump Freighters](https://wiki.eveuniversity.org/Jump_Freighters), helping reduce their fuel costs. Since most players primarily utilize Tech I ships and components, these modules have a much more specialized audience.

The [Fortizar](https://wiki.eveuniversity.org/Upwell_structure#Citadels) is a type of player-made structure used by corporations ("guilds" in other games) and by high-risk "null-sec" players for bases. These are expensive structures that relatively few players need or want to acquire.

For the sake of some simplicity in future analysis, I'm going to drop these two items. This leaves us with a retained study panel of **100 items**.

In [25]:
study_panel = study_panel.filter(~pl.col("type_id").is_in([34126, 35833]))

## 5. Retained-panel characteristics

With the panel selected, let's get a better sense of the markets we're keeping. We're going to compare their daily prices, trading activity, and price ranges over the available history, then look at how the panel is distributed across categories.

The historical calendar runs from the selection start through the latest observed date for the retained panel. Missing history rows are treated as zero activity for volume and history order count, while prices remain missing. Values in recorded rows are preserved.

### 5.1 Daily market-history histograms

For these distributions, I'm combining the items into seven broad groups: Ships, Modules, Ammunition and probes, Deployables, Drones, Minerals, and Rigs.

Each histogram shows the distribution of daily values within each group. The bars represent percentages of that group's positive, finite item-day values, using common logarithmically spaced bins. Zero, missing, and invalid values are excluded from those percentages, with their counts available in hover.

The relative daily price range is calculated as `(high − low) / VWAP`. This describes how far apart the day's highest and lowest transaction prices were relative to its average price.

You can click the legend to hide a group, or double-click it to isolate one.

In [26]:
retained_panel = study_panel.sort("type_id")
if retained_panel.is_empty() or retained_panel.get_column("type_id").n_unique() != retained_panel.height:
    raise ValueError("The retained study panel must contain unique items.")

retained_history_sql = """
select
    h.type_id,
    h.market_date,
    h.average as vwap,
    h.highest as daily_high,
    h.lowest as daily_low,
    h.volume,
    h.order_count
from int_forge_market_history as h
inner join notebook_retained_panel as p on h.type_id = p.type_id
where h.region_id = $region_id
    and h.market_date >= $history_start
"""

with open_analysis_connection() as con:
    con.register("notebook_retained_panel", retained_panel)
    retained_observed_history = con.execute(
        retained_history_sql,
        {"region_id": REGION_ID, "history_start": SELECTION_START},
    ).pl().with_columns(pl.lit(True).alias("_source_row_present"))

if retained_observed_history.is_empty():
    raise ValueError("No retained-panel history is available from the declared study start.")
if retained_observed_history.select(["type_id", "market_date"]).n_unique() != retained_observed_history.height:
    raise ValueError("Retained observed history must contain one source row per item-date.")

retained_cutoff_value = retained_observed_history.get_column("market_date").max()
retained_history_cutoff = date.fromisoformat(str(retained_cutoff_value))
if retained_history_cutoff < SELECTION_START:
    raise ValueError("The retained-panel history cutoff precedes the declared study start.")

retained_calendar_dates = pl.date_range(
    SELECTION_START, retained_history_cutoff, interval="1d", eager=True
).cast(pl.Date)
retained_calendar = retained_panel.join(
    pl.DataFrame({"market_date": retained_calendar_dates}), how="cross"
)
retained_panel_history = retained_calendar.join(
    retained_observed_history,
    on=["type_id", "market_date"],
    how="left",
    validate="1:1",
).with_columns(
    pl.when(pl.col("_source_row_present").is_null())
    .then(pl.lit(0))
    .otherwise(pl.col("volume"))
    .alias("volume"),
    pl.when(pl.col("_source_row_present").is_null())
    .then(pl.lit(0))
    .otherwise(pl.col("order_count"))
    .alias("order_count"),
)

if retained_panel_history.height != retained_panel.height * retained_calendar_dates.len():
    raise ValueError("The retained-panel calendar has an unexpected number of item-days.")

category_order = retained_panel.get_column("study_category").unique().sort().to_list()
# Fine-category colors are shared with the manufacturing view.
category_colors = CATEGORY_COLORS.copy()
unknown_histogram_categories = sorted(set(category_order) - set(HISTOGRAM_CATEGORY_GROUPS))
if unknown_histogram_categories:
    raise ValueError(f"Unknown histogram display categories: {unknown_histogram_categories}")


def relative_daily_dispersion(row):
    values = (row["vwap"], row["daily_high"], row["daily_low"])
    if any(value is None for value in values):
        return None
    try:
        vwap, daily_high, daily_low = (float(value) for value in values)
    except (TypeError, ValueError):
        return float("nan")
    if not all(math.isfinite(value) for value in (vwap, daily_high, daily_low)) or vwap <= 0:
        return float("nan")
    return (daily_high - daily_low) / vwap


def metric_value_status(value):
    if value is None:
        return "missing"
    try:
        numeric_value = float(value)
    except (TypeError, ValueError):
        return "nonfinite_invalid"
    if not math.isfinite(numeric_value):
        return "nonfinite_invalid"
    if numeric_value < 0:
        return "finite_invalid"
    if numeric_value == 0:
        return "zero"
    return "positive"


def log_histogram_summary(history, value_column, *, category_groups, group_order, bin_count):
    group_values = {group: [] for group in group_order}
    group_statuses = {
        group: {status: 0 for status in ("positive", "zero", "missing", "finite_invalid", "nonfinite_invalid")}
        for group in group_order
    }
    for row in history.select("study_category", value_column).iter_rows(named=True):
        category = row["study_category"]
        try:
            group = category_groups[category]
        except KeyError as error:
            raise ValueError(f"Unknown histogram display category: {category}") from error
        status = metric_value_status(row[value_column])
        group_statuses[group][status] += 1
        if status == "positive":
            group_values[group].append(float(row[value_column]))

    all_positive = [value for values in group_values.values() for value in values]
    if all_positive:
        log_min, log_max = math.log10(min(all_positive)), math.log10(max(all_positive))
        if math.isclose(log_min, log_max):
            log_min -= 0.5
            log_max += 0.5
    else:
        log_min, log_max = -0.5, 0.5
    log_edges = [log_min + (log_max - log_min) * step / bin_count for step in range(bin_count + 1)]
    bin_edges = [10**edge for edge in log_edges]

    histogram_rows = []
    for group, values in group_values.items():
        counts = [0] * bin_count
        for value in values:
            bin_index = min(bin_count - 1, max(0, int((math.log10(value) - log_min) / (log_max - log_min) * bin_count)))
            counts[bin_index] += 1
        total = sum(group_statuses[group].values())
        for bin_index, count in enumerate(counts):
            if count or not values:
                histogram_rows.append({
                    "histogram_group": group,
                    "bin_left": bin_edges[bin_index],
                    "bin_right": bin_edges[bin_index + 1],
                    "bin_center": math.sqrt(bin_edges[bin_index] * bin_edges[bin_index + 1]),
                    "count": count,
                    "percent_within_group": 0 if not values else count / len(values) * 100,
                    "positive_count": len(values),
                    "calendar_item_days": total,
                    "zero_count": group_statuses[group]["zero"],
                    "missing_count": group_statuses[group]["missing"],
                    "finite_invalid_count": group_statuses[group]["finite_invalid"],
                    "nonfinite_invalid_count": group_statuses[group]["nonfinite_invalid"],
                })
                if not values:
                    break
    return pl.DataFrame(histogram_rows), bin_edges


retained_panel_history = retained_panel_history.with_columns(
    pl.Series(
        "relative_daily_high_low_dispersion",
        [relative_daily_dispersion(row) for row in retained_panel_history.iter_rows(named=True)],
    )
)
# Common equal-width log10 bins; normalize each group by positive-finite item-days.
HISTOGRAM_BIN_COUNT = 30
histogram_specs = [
    ("vwap", "Daily VWAP (ISK/unit)"),
    ("volume", "Daily traded volume (units)"),
    ("order_count", "Daily history order count"),
    ("relative_daily_high_low_dispersion", "Relative daily price range: (high − low) / VWAP"),
]
histogram_summaries = {
    value_column: log_histogram_summary(
        retained_panel_history, value_column,
        category_groups=HISTOGRAM_CATEGORY_GROUPS,
        group_order=HISTOGRAM_GROUP_ORDER,
        bin_count=HISTOGRAM_BIN_COUNT,
    )
    for value_column, _ in histogram_specs
}

In [27]:
def plot_retained_distributions(
    summaries, *, specs, group_order, group_colors, title, y_label,
):
    fig = make_subplots(
        rows=2,
        cols=2,
        horizontal_spacing=0.10,
        vertical_spacing=0.14,
    )

    for metric_index, (value_column, axis_title) in enumerate(specs):
        row, col = divmod(metric_index, 2)
        histogram, _ = summaries[value_column]
        status_summaries = []
        for group in group_order:
            group_histogram = histogram.filter(pl.col("histogram_group") == group).sort("bin_center")
            if group_histogram.is_empty():
                continue
            status = group_histogram.row(0, named=True)
            if status["positive_count"] == 0:
                status_summaries.append(
                    f"{group}: 0 positive; {status['zero_count']} zero; {status['missing_count']} missing"
                )
                fig.add_trace(
                    go.Scatter(
                        x=[1],
                        y=[0],
                        mode="markers",
                        name=group,
                        legendgroup=group,
                        showlegend=metric_index == 0,
                        marker={
                            "color": group_colors[group],
                            "size": 9,
                            "symbol": "circle-open",
                            "line": {"color": group_colors[group], "width": 1},
                        },
                        customdata=[(
                            status["calendar_item_days"], status["zero_count"], status["missing_count"],
                            status["finite_invalid_count"], status["nonfinite_invalid_count"],
                        )],
                        hovertemplate=(
                            "<b>%{fullData.name}</b><br>No positive finite values"
                            "<br>Calendar item-days: %{customdata[0]}"
                            "<br>Zero: %{customdata[1]}"
                            "<br>Missing: %{customdata[2]}"
                            "<br>Finite invalid: %{customdata[3]}"
                            "<br>Non-finite invalid: %{customdata[4]}<extra></extra>"
                        ),
                    ),
                    row=row + 1,
                    col=col + 1,
                )
                continue
            bin_left = group_histogram.get_column("bin_left")
            bin_right = group_histogram.get_column("bin_right")
            customdata = [
                (
                    bin_row["bin_left"], bin_row["bin_right"], bin_row["count"],
                    bin_row["positive_count"], bin_row["calendar_item_days"], bin_row["zero_count"],
                    bin_row["zero_count"] / bin_row["calendar_item_days"] * 100,
                    bin_row["missing_count"], bin_row["missing_count"] / bin_row["calendar_item_days"] * 100,
                    bin_row["finite_invalid_count"], bin_row["nonfinite_invalid_count"],
                )
                for bin_row in group_histogram.iter_rows(named=True)
            ]
            fig.add_trace(
                go.Bar(
                    x=(bin_left + bin_right) / 2,
                    y=group_histogram.get_column("percent_within_group"),
                    width=bin_right - bin_left,
                    name=group,
                    legendgroup=group,
                    showlegend=metric_index == 0,
                    marker={
                        "color": group_colors[group],
                        "line": {"color": group_colors[group], "width": 0.75},
                    },
                    opacity=0.48,
                    customdata=customdata,
                    hovertemplate=(
                        "<b>%{fullData.name}</b><br>Bin left: %{customdata[0]:.4g}"
                        "<br>Bin right: %{customdata[1]:.4g}"
                        "<br>Positive finite values in bin: %{customdata[2]}"
                        "<br>Share of positive finite group values: %{y:.2f}%"
                        "<br>Positive finite: %{customdata[3]} of %{customdata[4]} item-days"
                        "<br>Zero: %{customdata[5]} (%{customdata[6]:.2f}%)"
                        "<br>Missing: %{customdata[7]} (%{customdata[8]:.2f}%)"
                        "<br>Finite invalid: %{customdata[9]}"
                        "<br>Non-finite invalid: %{customdata[10]}<extra></extra>"
                    ),
                ),
                row=row + 1,
                col=col + 1,
            )
        if status_summaries:
            fig.add_annotation(
                text="<br>".join(status_summaries),
                xref=f"x{metric_index + 1} domain" if metric_index else "x domain",
                yref=f"y{metric_index + 1} domain" if metric_index else "y domain",
                x=0.01,
                y=0.99,
                xanchor="left",
                yanchor="top",
                align="left",
                showarrow=False,
                bgcolor="rgba(255,255,255,0.8)",
                bordercolor="rgba(0,0,0,0.2)",
                borderwidth=1,
                font={"size": 10},
            )
        fig.update_xaxes(type="log", title_text=axis_title, row=row + 1, col=col + 1)
        fig.update_yaxes(title_text=y_label if col == 0 else None, row=row + 1, col=col + 1)

    fig.update_layout(
        title={
            "text": title,
            "x": 0.5,
            "xanchor": "center",
            "y": 0.98,
            "yanchor": "top",
        },
        barmode="overlay",
        width=1200,
        height=960,
        legend={
            "groupclick": "togglegroup",
            "itemclick": "toggle",
            "itemdoubleclick": "toggleothers",
            "orientation": "h",
            "y": 1.12,
            "yanchor": "bottom",
        },
        margin={"l": 70, "r": 30, "t": 190, "b": 70},
    )
    return fig

In [28]:
plot_retained_distributions(
    histogram_summaries, specs=histogram_specs, group_order=HISTOGRAM_GROUP_ORDER,
    group_colors=DISPLAY_GROUP_COLORS, title="Daily market distributions by group",
    y_label="Within-group share (%)",
)

A few differences between these groups are pretty apparent:

- **Daily VWAP:** Ships generally occupy the higher price ranges, while ammunition and probes are concentrated toward the lower end. Several groups also have multiple concentrations, reflecting the different types of items within them.
- **Daily traded volume:** Minerals stand out for the enormous quantities traded, followed by ammunition and probes. Ships generally trade in much smaller quantities.
- **Daily history order count:** Minerals also tend to have higher daily order counts, while ships generally occupy the lower ranges. Modules, drones, rigs, and ammunition/probes overlap quite a bit.
- **Relative daily price range:** The groups overlap substantially here, although rigs appear more concentrated toward smaller relative ranges. Several groups also have long tails of days with unusually large high–low differences.

These differences give us some useful context for later anomaly analysis. What looks like unusually high volume or a large price movement for one item might be fairly ordinary for another.

**TODO:** Reduce the whitespace between the plots, especially around the title and top legend.

### 5.2 Selection-period category composition

Finally, let's compare how much of the retained panel each category accounts for by item count and by estimated traded ISK during the original selection window.

These use the more detailed categories assigned earlier, rather than the seven broad groups used in the histograms.

In [29]:
retained_selection = retained_panel.join(
    proposed_panel.select("type_id", "total_estimated_traded_isk"),
    on="type_id",
    how="left",
    validate="1:1",
)
if retained_selection.get_column("total_estimated_traded_isk").null_count():
    raise ValueError("Each retained item needs a selection-period estimated traded ISK total.")

category_composition = retained_selection.group_by("study_category").agg(
    pl.len().alias("retained_item_count"),
    pl.col("total_estimated_traded_isk").sum().alias("selection_estimated_traded_isk"),
).with_columns(
    (pl.col("retained_item_count") / pl.col("retained_item_count").sum() * 100).alias("retained_item_share_pct"),
    (
        pl.col("selection_estimated_traded_isk")
        / pl.col("selection_estimated_traded_isk").sum()
        * 100
    ).alias("selection_estimated_traded_isk_share_pct"),
).with_columns(
    pl.col("study_category").replace_strict(
        {category: index for index, category in enumerate(category_order)}
    ).alias("category_sort_order")
).sort("category_sort_order", descending=True)

if not math.isclose(category_composition.get_column("retained_item_share_pct").sum(), 100.0):
    raise ValueError("Retained-item category shares must sum to 100%.")
if not math.isclose(category_composition.get_column("selection_estimated_traded_isk_share_pct").sum(), 100.0):
    raise ValueError("Selection-period estimated traded ISK category shares must sum to 100%.")

In [30]:
def plot_retained_composition(
    composition, *, colors, title, panel_titles, x_label,
):
    fig = make_subplots(
        rows=1,
        cols=2,
        shared_yaxes=True,
        subplot_titles=panel_titles,
        horizontal_spacing=0.16,
    )
    for column, label, subplot_column in [
        ("retained_item_share_pct", "Retained-item share", 1),
        (
            "selection_estimated_traded_isk_share_pct",
            "Selection-period estimated traded ISK share",
            2,
        ),
    ]:
        fig.add_trace(
            go.Bar(
                x=composition.get_column(column),
                y=composition.get_column("study_category"),
                orientation="h",
                marker={"color": [colors[category] for category in composition.get_column("study_category")]},
                customdata=composition.select(
                    "retained_item_count", "selection_estimated_traded_isk"
                ).rows(),
                hovertemplate=(
                    "<b>%{y}</b><br>Share: %{x:.2f}%"
                    "<br>Retained items: %{customdata[0]}"
                    "<br>Selection-period estimated traded ISK: %{customdata[1]:,.0f}"
                    "<extra></extra>"
                ),
                showlegend=False,
                name=label,
            ),
            row=1,
            col=subplot_column,
        )
        fig.update_xaxes(title_text=x_label, ticksuffix="%", row=1, col=subplot_column)

    fig.update_yaxes(showticklabels=False, row=1, col=2)
    fig.update_layout(
        title=title,
        width=1200,
        height=max(550, 35 * composition.height + 180),
        margin={"l": 250, "r": 40, "t": 100, "b": 70},
    )
    return fig

In [31]:
plot_retained_composition(
    category_composition, colors=CATEGORY_COLORS, title="Retained-panel composition",
    panel_titles=("Share of retained items", "Share of estimated traded value"),
    x_label="Share (%)",
)

Ammunition and probes, cruisers and battlecruisers, and rigs account for substantial shares of the retained item types. The picture looks quite different when we weight the categories by estimated traded ISK.

The seven minerals make up just **7% of the panel's items**, but account for approximately **48% of its estimated traded value** during the selection window. Freighters also account for a much larger share of traded value than their small number of item types might suggest.

So although the panel contains a fairly broad mix of products, a large share of its traded ISK is concentrated in minerals and expensive ships.

**TODO:** Add a third panel showing each category's share of total history order count, since that was the other measure used to select the panel.

## 6. Production graph

So far, we've mostly looked at these item markets individually or grouped them into categories. But many of them are also connected through manufacturing.

The graph below connects our retained items to their direct production inputs and downstream manufacturing outputs. The seven minerals appear separately from the other external inputs, while products outside the study panel are grouped by their Tech/Meta classification.

In [32]:
import sys

repo_root = find_repo_root()
experiments_root = repo_root / "experiments"
if str(experiments_root) not in sys.path:
    sys.path.insert(0, str(experiments_root))


from market_analysis.helpers.production_graph import (
    prepare_production_graph,
    layout_production_graph,
)


def manufacturing_links(blueprints, reference_types):
    links_by_pair = {}

    for blueprint_key in sorted(blueprints, key=int):
        blueprint_id = int(blueprint_key)
        blueprint = blueprints[blueprint_key]
        blueprint_type = reference_types.get(str(blueprint_id))
        if not isinstance(blueprint_type, dict) or blueprint_type.get("published") is not True:
            continue
        if not isinstance(blueprint, dict) or not isinstance(blueprint.get("activities"), dict):
            raise ValueError(f"Invalid activities for blueprint {blueprint_id}.")
        manufacturing = blueprint["activities"].get("manufacturing")
        if manufacturing is None:
            continue
        if not isinstance(manufacturing, dict):
            raise ValueError(f"Invalid manufacturing activity for blueprint {blueprint_id}.")
        materials = manufacturing.get("materials")
        products = manufacturing.get("products")
        if materials is None or materials == {} or products is None or products == {}:
            continue
        if not isinstance(materials, dict) or not isinstance(products, dict):
            raise ValueError(f"Invalid manufacturing recipe for blueprint {blueprint_id}.")

        inputs = []
        for material_key, material in materials.items():
            input_type_id = material.get("type_id") if isinstance(material, dict) else None
            input_quantity = material.get("quantity") if isinstance(material, dict) else None
            if (
                type(input_type_id) is not int
                or input_type_id <= 0
                or str(input_type_id) != material_key
                or type(input_quantity) is not int
                or input_quantity <= 0
            ):
                raise ValueError(f"Invalid material {material_key} for blueprint {blueprint_id}.")
            inputs.append((input_type_id, input_quantity))

        outputs = []
        for product_key, product in products.items():
            output_type_id = product.get("type_id") if isinstance(product, dict) else None
            output_quantity = product.get("quantity") if isinstance(product, dict) else None
            if (
                type(output_type_id) is not int
                or output_type_id <= 0
                or str(output_type_id) != product_key
                or type(output_quantity) is not int
                or output_quantity <= 0
            ):
                raise ValueError(f"Invalid product {product_key} for blueprint {blueprint_id}.")
            outputs.append((output_type_id, output_quantity))

        for input_type_id, input_quantity in inputs:
            for output_type_id, output_quantity in outputs:
                pair = (input_type_id, output_type_id)
                recipe = (input_quantity, output_quantity)
                previous_recipe = links_by_pair.get(pair)
                if previous_recipe is not None:
                    if previous_recipe != recipe:
                        raise ValueError(
                            f"Conflicting manufacturing recipes for {input_type_id} -> {output_type_id}."
                        )
                    raise ValueError(
                        f"Duplicate manufacturing recipe for {input_type_id} -> {output_type_id}."
                    )
                links_by_pair[pair] = recipe

    return pl.DataFrame(
        [
            {
                "input_type_id": input_type_id,
                "output_type_id": output_type_id,
                "input_quantity": input_quantity,
                "output_quantity": output_quantity,
            }
            for (input_type_id, output_type_id), (input_quantity, output_quantity)
            in sorted(links_by_pair.items())
        ],
        schema={
            "input_type_id": pl.Int64,
            "output_type_id": pl.Int64,
            "input_quantity": pl.Int64,
            "output_quantity": pl.Int64,
        },
    )


production_graph_links = manufacturing_links(blueprints, reference_types)
production_graph_model = prepare_production_graph(
    production_graph_links, study_panel, reference_types, meta_groups, MINERAL_IDS,
    category_palette=CATEGORY_COLORS,
)
production_positions, production_regions = layout_production_graph(production_graph_model)
production_graph_model = {
    **production_graph_model, "positions": production_positions, "layout_regions": production_regions,
}

In [33]:
from market_analysis.helpers.production_graph import (
    build_production_figure,
    production_graph_html,
)

In [34]:
HTML(production_graph_html(build_production_figure(
    production_graph_model, production_positions, production_regions,
    title="Study-panel manufacturing relationships",
    direction_key="Material → product", legend_title="Item category",
), production_graph_model))

The minerals connect to many of our retained products, giving otherwise fairly different markets a shared set of production inputs. Some products also require additional materials beyond these seven minerals.

Several retained products are themselves used to manufacture more advanced goods, including Tech II items. So these markets aren't just connected to the supply of raw materials; they're also connected to demand from downstream manufacturing.

These relationships give us something useful to examine in later analysis. For example, do products that share manufacturing inputs experience similar price movements? When a mineral's price changes substantially, do related product markets move with it, and how long does that take?

## 7. Export study seeds

With the exclusions above applied, I export the retained IDs and study categories to `transformation/seeds/forge_study_panel.csv` and production relationships to `transformation/seeds/production_links.csv`. This cell rewrites the seeds each time it runs. If either seed changes, rebuild the downstream dbt models with `mise run transform:build` before using the frozen panel in later notebooks.

In [35]:
from pathlib import Path

seed = study_panel.select("type_id", "study_category").sort("type_id")
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "transformation" / "dbt_project.yml").is_file()
)

production_links = manufacturing_links(blueprints, reference_types)

seed_output_path = globals().get("SEED_OUTPUT_PATH")
seed_path = Path(seed_output_path) if seed_output_path else repo_root / "transformation" / "seeds" / "forge_study_panel.csv"
production_links_output_path = globals().get("PRODUCTION_LINKS_OUTPUT_PATH")
production_links_path = (
    Path(production_links_output_path)
    if production_links_output_path
    else repo_root / "transformation" / "seeds" / "production_links.csv"
)

seed.write_csv(seed_path, line_terminator="\n")
production_links.write_csv(production_links_path, line_terminator="\n")
seed_display_path = (
    str(seed_path.relative_to(repo_root))
    if seed_path.is_relative_to(repo_root)
    else str(seed_path)
)

final_panel = proposed_panel.join(
    study_panel.select("type_id", "study_category"), on="type_id", how="inner", validate="1:1"
)
export_summary = pl.DataFrame({
    "Frozen panel items": [study_panel.height],
    "Curated minerals": [study_panel.filter(pl.col("type_id").is_in(MINERAL_IDS)).height],
    "In both ranked lists": [final_panel.filter(
        pl.col("in_order_count_top_50") & pl.col("in_estimated_isk_top_50")
    ).height],
    "Unavailable names": [study_panel.get_column("type_name").null_count()],
    "Generated seed": [seed_display_path],
})
export_summary